## 01. 회귀 기반 추천 시스템

회귀 기반 추천은 추천 문제를 “점수 예측 문제”로 바꾸어 해결하는 방식이다.

### 회귀 기반 추천 시스템흐름

1. 추천 후보를 준비한다.
2. 사용자 조건과 후보 정보를 모델에 넣는다.
3. 모델이 후보별 예상 점수를 예측한다.
4. 예측 점수가 높은 후보를 위로 정렬한다.
5. 정렬 결과의 상위 항목을 추천 결과로 사용한다.

예를 들어 여행지 추천에서는 사용자의 성별, 연령대, 여행 스타일, 이동수단, 후보 여행지 같은 정보를 입력으로 넣고, 해당 여행지에 대한 예상 만족도 점수를 예측한다.  
이후 예측 만족도가 높은 여행지를 위에서부터 추천한다.

---

**왜 회귀 모델로 추천을 만들 수 있는가?**

회귀 모델은 숫자를 예측하는 모델이다.  
추천 문제에서 예측하고 싶은 값을 “만족도”, “평점”, “구매 가능 점수”, “클릭 가능 점수”처럼 숫자로 정의하면 회귀 모델을 추천에 활용할 수 있다.

이 노트북에서는 `DGSTFN` 만족도 점수를 예측한다.  
따라서 모델이 출력하는 값은 “이 사용자가 이 방문지를 갔을 때 예상되는 만족도”로 해석한다.

---

**사용하는 모델: CatBoostRegressor**

CatBoostRegressor는 여러 개의 트리를 순서대로 학습하면서 오차를 줄여가는 부스팅 계열 회귀 모델이다.

범주형 feature를 비교적 편하게 다룰 수 있어 여행 지역, 이동수단, 여행 목적처럼 문자열이나 범주형 값이 많은 추천 예제에 잘 맞는다.

---

**확인할 평가 지표**

- MAE: 예측 만족도와 실제 만족도의 차이를 절댓값으로 평균낸 값이다. 낮을수록 좋음.
- RMSE: 큰 오차에 더 민감한 회귀 오차 지표이다. 낮을수록 좋음.
- R2: 평균값으로 예측하는 것보다 모델이 target 변화를 얼마나 더 잘 설명하는지 보는 지표이다. 1에 가까울수록 좋음.

추천 시스템에서는 평가 지표가 좋다고 바로 좋은 서비스가 되는 것은 아니다.  
모델이 예측한 점수를 기준으로 후보를 정렬했을 때 실제 사용자가 만족할 만한 추천 목록이 만들어지는지도 함께 봐야 한다.

---

**이 방식의 한계**

회귀 기반 추천은 “사용자와 후보 항목을 넣으면 점수를 예측한다”는 구조가 단순해서 이해하기 쉽다.  
하지만 사용자의 장기 행동 이력, 다른 사용자와의 유사성, 최신 선호 변화까지 모두 자동으로 반영하는 고도화된 추천 시스템은 아니다.

따라서 이 노트북은 추천 시스템 전체를 완성하는 단원이라기보다, 추천 시스템의 기본 구조인 “후보 생성 → 점수 예측 → 정렬”을 이해하기 위한 첫 번째 실습이다.


## 02. 추천 시스템을 가장 쉽게 시작하는 방법

추천 시스템을 처음 배울 때는 “복잡한 알고리즘”보다 “추천이 만들어지는 구조”를 먼저 보는 것이 좋다.

회귀 기반 추천은 추천 문제를 다음처럼 단순화한다.

```text
사용자 조건 + 후보 항목 정보 -> 예상 점수 예측 -> 점수순 정렬 -> 추천
```

여기서 중요한 점은 모델이 바로 “추천 목록”을 만들어주는 것이 아니라는 점이다.  
모델은 후보별 예상 점수를 예측하고, 우리는 그 점수를 기준으로 후보를 정렬해 추천 목록을 만든다.

이번 실습에서는 여행 데이터를 사용해 다음 질문에 답한다.

> 이 사용자 조건에서 어떤 방문지가 높은 만족도를 받을 가능성이 있는가?


## 03. 제주도 여행데이터

https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=data&dataSetSn=71584

이 데이터는 제주도 여행자의 기본 정보, 여행 스타일, 여행 동기, 방문지, 이동수단, 만족도 점수를 담고 있다.
여행자 조건과 후보 방문지 정보를 입력으로 넣고, `DGSTFN` 만족도 점수를 예측한 뒤 점수가 높은 방문지를 추천한다.

**컬럼 구성**

| 컬럼 | 의미 | 추천 모델에서의 역할 |
|---|---|---|
| `GENDER` | 여행자 성별 | 입력 feature |
| `AGE_GRP` | 여행자 연령대 코드 | 입력 feature |
| `TRAVEL_STYL_1` ~ `TRAVEL_STYL_8` | 여행 스타일 관련 설문 응답 코드값 | 입력 feature |
| `TRAVEL_MOTIVE_1` | 주요 여행 동기 코드 | 입력 feature |
| `TRAVEL_COMPANIONS_NUM` | 여행 동반자 수 | 입력 feature |
| `VISIT_AREA_NM` | 방문지 이름 | 추천 후보이자 입력 feature |
| `MVMN_NM` | 이동수단 이름 | 입력 feature |
| `DGSTFN` | 방문지 만족도 점수 | 예측할 target |

**학습 포인트**

- `GENDER`, `VISIT_AREA_NM`, `MVMN_NM`은 문자열 범주형 feature이다.
- `AGE_GRP`, `TRAVEL_STYL_*`, `TRAVEL_MOTIVE_1`, `TRAVEL_COMPANIONS_NUM`은 숫자로 보이지만 실제 의미는 코드값에 가까운 범주형 feature이다.
- `DGSTFN`은 모델이 맞혀야 하는 정답값이다. 이 값이 높게 예측되는 방문지를 추천 후보 상위에 배치한다.
- 따라서 이 예제는 “사용자 조건 + 후보 방문지 → 예상 만족도 점수”를 예측하는 회귀 기반 추천 문제이다.


## 04. 실습 환경 준비

- 라이브러리 import가 끝나면 데이터 처리, 모델 학습, 평가에 필요한 도구를 사용할 수 있음.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 05. 데이터 로드와 기본 확인

- 데이터 로드는 모델 학습 전에 컬럼 의미, 행/열 크기, target 분포를 확인하는 출발점임.


In [ ]:
travel_df = pd.read_csv('data/travel.csv')
travel_df.head()


## 06. 범주형 feature 타입 정리

- 여행 스타일과 동기 컬럼을 모델에 전달하기 전에 일관된 정수형 범주 값으로 정리함.


## 07. 데이터 구조 확인

- 자료형이 숫자인지 범주형인지, 결측치가 있는지에 따라 인코딩과 결측치 처리 방법이 달라짐.


In [ ]:
travel_df.info()


## 08. 기초 통계 확인

- 평균과 중앙값 차이, 최솟값/최댓값 범위를 보면 스케일링이나 이상치 처리가 필요한지 판단할 수 있음.


In [ ]:
travel_df.describe()


## 09. 분포와 집계 확인

- 분포가 치우치면 stratify, 평가지표 선택, 샘플링 전략이 달라질 수 있음.


In [ ]:
travel_df['TRAVEL_COMPANIONS_NUM'].value_counts()


## 10. 데이터 일부 확인

- head 결과는 값의 형태와 컬럼 의미를 빠르게 확인하는 용도이며, 전체 품질 판단은 info/describe와 함께 봐야 함.


In [ ]:
print(travel_df.columns)

display(travel_df.head())


## 11. CatBoost에 범주형 feature로 전달 입력 컬럼 목록 조회

## 12. 학습/평가 데이터 분리

## 13. CatBoost 회귀 추천 모델 학습

CatBoost는 범주형 feature 처리를 지원하므로 여행 스타일처럼 문자/범주 성격이 강한 데이터에 활용하기 좋다.

CatBoostRegressor는 여행자 정보와 후보 방문지 정보를 입력받아 `DGSTFN` 만족도 점수를 예측한다.
즉, 모델의 출력값은 class가 아니라 숫자 점수이다.


- `n_estimators`: 순서대로 만들 트리 개수이다. 많을수록 복잡한 패턴을 학습할 수 있지만 시간이 오래 걸릴 수 있다.
- `depth`: 각 트리의 깊이이다. 깊을수록 복잡한 규칙을 만들 수 있지만 과대적합 위험이 커질 수 있다.
- `learning_rate`: 새 트리가 이전 오차를 얼마나 크게 보정할지 정하는 값이다.
- `loss_function='RMSE'`: 모델이 줄이려고 하는 손실 기준이다.


## 14. 학습 과정 시각화

학습 로그를 그래프로 그리면 반복 횟수가 늘어날 때 훈련 오차와 검증 오차가 어떻게 바뀌는지 볼 수 있다.

- Train RMSE: 모델이 학습 데이터에 얼마나 맞는지 보여준다.
- Validation RMSE: 처음 보는 데이터에 대한 예측 오차를 보여준다.
- Train RMSE만 낮고 Validation RMSE가 높으면 학습 데이터를 과하게 외운 상태일 수 있다.


## 15. Feature 중요도 확인

Feature 중요도는 모델이 예측할 때 어떤 입력 변수를 많이 활용했는지 보여준다.

추천 시스템에서는 “왜 이 항목이 추천되었는가?”를 설명해야 하는 경우가 많다.  
예를 들어 방문지 이름, 이동수단, 여행 스타일 중 어떤 정보가 만족도 예측에 많이 쓰였는지 확인하면 모델의 판단 근거를 조금 더 설명할 수 있다.

CatBoost 같은 트리 기반 모델에서 중요도는 보통 분기와 손실 감소에 얼마나 기여했는지 기준으로 계산된다.  
중요도가 높다고 해서 반드시 만족도의 원인이라는 뜻은 아니며, 모델이 예측에 자주 참고한 변수로 해석하는 것이 안전하다.


## 16. 예측과 점수 확인

회귀 기반 추천에서는 모델이 예측한 숫자 점수가 추천 정렬의 기준이 된다.  
따라서 추천 함수를 만들기 전에, 이 예측 점수가 어느 정도 믿을 만한지 먼저 확인해야 한다.

여기서 예측 점수는 확률이 아니다.  
`DGSTFN` 만족도 점수를 예측한 값이므로 “이 후보 방문지에 대해 모델이 예상한 만족도”로 해석한다.

- MAE: 평균적으로 몇 점 정도 틀리는지 직관적으로 볼 수 있음.
- RMSE: 큰 오차를 더 강하게 반영함.
- R2: 모델이 target 변화를 얼마나 설명하는지 확인함.

추천에서는 단일 샘플 하나를 맞혔는지보다, 여러 후보의 예측 점수를 비교했을 때 의미 있는 순서가 만들어지는지가 중요하다.


### 추천 함수

추천 함수는 “후보 장소를 하나씩 바꿔 넣어 예측 만족도를 계산하고, 점수가 높은 장소를 정렬하는 함수”이다.

처리 흐름은 다음과 같다.

1. 사용자 조건은 고정한다.
2. 후보 방문지만 하나씩 바꾼다.
3. 각 후보 방문지에 대한 예상 만족도를 예측한다.
4. 예측 만족도가 높은 순서로 정렬해 상위 N개를 추천한다.

이 과정에서 모델은 매번 “사용자 조건 + 후보 방문지” 조합에 대해 만족도 점수를 예측한다.  
즉, 추천 함수는 모델의 `predict()` 결과를 여러 번 모아서 순위를 만드는 역할을 한다.


## 17. 추천 후보가 될 전체 방문지 목록 확인

## 18. 추천 함수에 넣을 사용자 입력값 구조 확인

## 19. 예측 점수 기반 추천 결과 생성

이제 학습된 CatBoost 모델을 추천 함수 안에서 사용한다.

중요한 해석 포인트는 다음이다.

- `user_input`: 한 명의 사용자 조건과 후보 방문지 자리를 담은 입력값임.
- `candidate_input[-2]`: 방문지 이름이 들어가는 위치임.
- `dgstfn_pred`: 모델이 예측한 예상 만족도 점수임.
- 추천 결과는 `dgstfn_pred`가 높은 순서로 정렬됨.

따라서 최종 표의 첫 번째 행은 “현재 사용자 조건에서 모델이 가장 높은 만족도를 예상한 방문지”로 해석한다.


## 20. 여러 사용자 조건으로 예측해보기

하나의 사용자 입력만 확인하면 추천 함수가 실제로 사용자 조건에 따라 달라지는지 보기 어렵다.

이번 셀에서는 평가 데이터에서 사용자 조건 몇 개를 가져와, 여러 후보 방문지에 대한 예상 만족도를 비교한다.  
이렇게 보면 “사용자 조건이 달라지면 같은 방문지라도 예측 만족도가 달라질 수 있다”는 점을 확인할 수 있다.

주의할 점은 이번 셀이 **전체 방문지 중 최종 추천을 다시 구하는 셀은 아니라는 것**이다.  
아래 코드에서는 `value_counts().head(5)`로 데이터에 자주 등장한 방문지 5개만 골라 비교한다.

따라서 이 셀의 결과는 다음처럼 해석한다.

```text
전체 방문지 중 추천 결과가 아니라,
선택한 후보 5개 안에서 사용자별 예상 만족도를 비교한 결과이다.
```

확인할 내용은 다음이다.

- 행: 서로 다른 사용자 조건
- 열: 비교 대상으로 선택한 후보 방문지 5개
- 값: CatBoost 모델이 예측한 예상 만족도 점수

점수가 높은 후보일수록 현재 사용자 조건에서 우선 추천될 가능성이 높다.  
다만 후보를 5개로 제한했기 때문에, 이 표에 없는 다른 방문지가 실제 전체 추천에서는 더 높은 점수를 받을 수도 있다.
